Проверенная редакция 18.11.2026. Исторические результаты сохранены в results/. Корпус и выборки фиксированы. Новые запуски пишут в отдельные рабочие каталоги. Проверка: scripts/audit_release.py. Полные модельные прогоны в этой редакции не повторялись.

# Lossy-RAG: Nomic ONLY (Matryoshka vs PCA)

**Run in Jupyter: Kernel -> Restart & Run All.**

This notebook is a targeted follow-up to the finished BGE experiments. It will NOT
recompute BGE Small/Base/Large or the BGE PCA experiments.

- Reuses the already prepared full SQuAD corpus in `D:/Lossy-RAG/data`.
- Uses the same 500 original questions (and existing paired paraphrases) as previous runs.
- Tests Nomic 768 dimensions (full), then Matryoshka and PCA 64/128/256/512.
- PCA fit is repeated for seeds 42, 43, and 44.
- Includes progress messages, resumable document embedding, cached search, and saved CSV results.
- Requires access to Hugging Face for Nomic model weights if not already cached.
- Automatically upgrades Transformers and Sentence-Transformers if too old to load Nomic natively.

**IMPORTANT:** Do not delete the `cache`, `data`, or previous results folders.



In [ ]:
# 0. Nomic-only settings. Existing BGE files remain untouched.
import os
from pathlib import Path

# Корень текущего репозитория или явная переменная окружения.
# Root is explicit, or the nearest repository above the notebook working directory.
_candidates = [Path.cwd(), *Path.cwd().parents]
_detected = next((p for p in _candidates if (p / 'data/build_data.py').is_file()), None)
_value = os.environ.get('LOSSY_RAG_ROOT')
if not _value and _detected is None:
    raise RuntimeError('Open Jupyter inside Lossy-RAG or set LOSSY_RAG_ROOT to the repository root.')
ROOT = Path(_value).expanduser().resolve() if _value else _detected
BLOCK1_DIR = ROOT / 'experiments/block1'
import sys

sys.path.insert(0, str(BLOCK1_DIR / 'scripts'))
from retrieval import array_hash, stable_topk_positions

ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR = ROOT / "data"                 # сюда создадим корпус, если другие расположения не найдены
AUTO_PREPARE_DATA = False                 # автоматическая подготовка полного SQuAD, если CSV отсутствуют
PREVIOUS_RESULTS = ROOT / "lossy_rag_intermediate_results"
PREVIOUS_FINAL = ROOT / "lossy_rag_final_results"
RESULT_DIR = ROOT / "lossy_rag_final_results_safe"
HF_HOME = ROOT / "hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

SEED = 42
N_QUESTIONS = 500
MIN_REQUIRED_CHUNKS = 80_000          # не допускаем 350 фрагментов вместо полного корпуса
K_VALUES = (1, 3, 5, 10)

# Рекомендуемый безопасный старт: проверка кэша, BGE Small и PCA
RUN_BGE_SMALL = False
RUN_BGE_BASE = False                # финальное сравнение моделей
RUN_BGE_LARGE = False               # финальное сравнение моделей
RUN_PCA_BGE = False
PCA_DIMS = (64, 128, 192, 256)
PCA_SEEDS = (42, 43, 44)           # устойчивость к выбору обучающей выборки
PCA_FIT_SAMPLE = 4096

RUN_PARAPHRASES = True             # попарная проверка реальных перефраз
RUN_NOMIC = True                   # Matryoshka и PCA на одной модели
NOMIC_DIMS = (64, 128, 256, 512)
RUN_PCA_NOMIC = True              # сравнить Nomic PCA vs Matryoshka, если RUN_NOMIC=True

DEVICE = "auto"                  # auto, cpu или cuda
THREADS = 4                      # ограничение потоков BLAS при поиске и PCA
ENCODE_BATCH_SIZE = 16           # на CPU можно 8-16, на GPU 32-64
QUERY_BLOCK = 16                 # 16 запросов на блок -> частые обновления статуса
DOC_BLOCK = 4096                 # ограничение пиковой оперативной памяти
PRINT_EVERY_BLOCKS = 2

BGE_MODELS = {
    "small": "BAAI/bge-small-en-v1.5",
    "base": "BAAI/bge-base-en-v1.5",
    "large": "BAAI/bge-large-en-v1.5",
}
NOMIC_MODEL = "nomic-ai/nomic-embed-text-v1.5"
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
NOMIC_QUERY_PREFIX = "search_query: "
NOMIC_DOC_PREFIX = "search_document: "

print("NOMIC ONLY: BGE encoding and BGE PCA are disabled.")
print("Данные:", DATA_DIR)
print("Результаты:", RESULT_DIR)
AUTO_INSTALL_MISSING = True   # ставить только отсутствующие библиотеки в окружение этого Jupyter
CONTINUE_AFTER_MODEL_ERROR = True  # отдельная модель может упасть, сводка предыдущих этапов сохранится

In [ ]:
# 1. Resolve missing requirements and use native Nomic model support.
# This cell runs before importing Transformers/Sentence-Transformers.
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as package_version

packages = {
    'numpy':'numpy', 'pandas':'pandas', 'scipy':'scipy',
    'sklearn':'scikit-learn', 'matplotlib':'matplotlib',
    'threadpoolctl':'threadpoolctl', 'torch':'torch',
    'einops':'einops', 'pyarrow':'pyarrow', 'spacy':'spacy',
    'huggingface_hub':'huggingface_hub', 'tqdm':'tqdm',
}
missing = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]
try:
    from packaging.version import Version
except ImportError:
    missing.append('packaging')
    Version = None

required_native = {
    'transformers': '5.5.0',
    'sentence-transformers': '5.3.0',
}
upgrade = []
for pkg, min_v in required_native.items():
    try:
        have = package_version(pkg)
    except PackageNotFoundError:
        have = None
    if have is None or (Version is not None and Version(have) < Version(min_v)):
        upgrade.append(f'{pkg}>={min_v}')
    print(f'{pkg}: installed={have or "missing"}, required>={min_v}', flush=True)

to_install = missing + upgrade
if to_install:
    if 'transformers' in sys.modules or 'sentence_transformers' in sys.modules:
        raise RuntimeError('Libraries were imported before the upgrade. Restart the kernel and Run All.')
    print('Installing/upgrading:', to_install, flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', *to_install], check=True)
    importlib.invalidate_caches()
    from packaging.version import Version
    for pkg, min_v in required_native.items():
        actual = package_version(pkg)
        assert Version(actual) >= Version(min_v), f'{pkg}={actual} < {min_v}'
    print('Dependencies ready. If a module reload error appears later, Kernel -> Restart & Run All.', flush=True)
else:
    print('Compatible Nomic libraries already installed; no pip changes required.', flush=True)
print('Python:', sys.executable)
print('Workspace:', ROOT)


In [ ]:
# 1. Подготовка, логирование и ограничение числа вычислительных потоков
import gc
import hashlib
import json
import platform
import time
import traceback
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits

for sub in ("cache", "rankings", "tables", "figures", "pca"):
    (RESULT_DIR / sub).mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
LOG_PATH = RESULT_DIR / "run_log.txt"

def log(msg):
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")


def json_atomic(path, obj):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)


def normalize(x):
    a = np.asarray(x, dtype=np.float32)
    denom = np.linalg.norm(a, axis=1, keepdims=True)
    return a / np.maximum(denom, 1e-12)


def layer_norm(x):
    x = np.asarray(x, dtype=np.float32)
    z = x - x.mean(axis=1, keepdims=True)
    return z / np.sqrt(np.mean(z*z, axis=1, keepdims=True) + 1e-5)


def signature(ids, texts):
    h = hashlib.sha256()
    for i, t in zip(ids, texts):
        h.update(str(int(i)).encode())
        h.update(b"\0")
        h.update(str(t).encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def checked_vectors(path, n_expected, d_expected=None):
    p = Path(path)
    if not p.is_file():
        return None
    try:
        a = np.load(p, mmap_mode="r")
        if a.ndim != 2 or len(a) != n_expected or a.dtype != np.float32:
            return None
        if d_expected is not None and a.shape[1] != d_expected:
            return None
        for start in range(0, len(a), 4096):
            block = np.asarray(a[start:start+4096])
            if not np.isfinite(block).all() or (np.linalg.norm(block, axis=1) < 1e-8).any():
                return None
        return a
    except (ValueError, OSError):
        return None

try:
    import torch
    DEVICE_USED = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
    torch.set_num_threads(max(1, THREADS))
except ImportError:
    DEVICE_USED = "cpu"
    torch = None
log(f"Python {platform.python_version()}, device={DEVICE_USED}, threads={THREADS}")
# Журнал ошибок этапов; даже при ошибке модели готовые CSV остаются на диске.
# The previous notebook left a Nomic error file. Archive it so a successful
# Nomic-only run cannot be mistaken for the previous failed attempt.
old_errors = RESULT_DIR / 'stage_errors.json'
if old_errors.exists():
    previous = RESULT_DIR / 'stage_errors_previous_run.json'
    if previous.exists():
        previous.unlink()
    old_errors.replace(previous)
STAGE_ERRORS = []
def register_error(stage, exception):
    STAGE_ERRORS.append({'stage': stage, 'error_type': type(exception).__name__, 'message': str(exception)})
    log(f'ОШИБКА {stage}: {type(exception).__name__}: {exception}')
    log(traceback.format_exc())
    json_atomic(RESULT_DIR / 'stage_errors.json', STAGE_ERRORS)
    if not CONTINUE_AFTER_MODEL_ERROR:
        raise exception

In [ ]:
# 2. Поиск готового полного корпуса (или подготовка SQuAD при отсутствии).
#    Исходный ноутбук считал, что CSV обязательно лежат в D:/Lossy-RAG/data.
#    Эта версия сначала ищет их там, где предыдущие ноутбуки обычно сохраняли файлы.
import importlib.util
import subprocess
import sys
from urllib.request import Request, urlopen

_REQUIRED = ("chunks.csv", "questions.csv", "answer_chunks.csv")
_SOURCE_BASE = "https://raw.githubusercontent.com/Arnoochka/Lossy-RAG/198aa66b0003ef759ecca6cd779176913df02fbc/data/"


def prepared_data_exists(folder):
    folder = Path(folder)
    return folder.is_dir() and all((folder / x).is_file() for x in _REQUIRED)


def likely_corpus_paths():
    cwd = Path.cwd()
    home = Path.home()
    candidates = [
        Path(os.environ['LOSSY_RAG_DATA_DIR']) if os.environ.get('LOSSY_RAG_DATA_DIR') else DATA_DIR,
        DATA_DIR,
        ROOT / "lossy_rag_data",
        ROOT / "data",
        cwd / "lossy_rag_data",
        cwd / "data",
        cwd.parent / "lossy_rag_data",
        cwd.parent / "data",
        home / "lossy_rag_data",
        home / "Lossy-RAG" / "data",
        home / "Downloads" / "lossy_rag_data",
        home / "Documents" / "lossy_rag_data",
        home / "Desktop" / "lossy_rag_data",
        home / "Downloads" / "Lossy-RAG" / "data",
        home / "Documents" / "Lossy-RAG" / "data",
        home / "Desktop" / "Lossy-RAG" / "data",
        home / "OneDrive" / "Documents" / "lossy_rag_data",
        home / "OneDrive" / "Desktop" / "lossy_rag_data",
        Path("D:/Lossy-RAG/data"),
        Path("D:/Lossy-RAG/lossy_rag_data"),
        Path("D:/lossy_rag_data"),
    ]
    # Ограниченный поиск: первый уровень внутри обычных папок, не всё дерево диска C:.
    for parent in (cwd, home / "Downloads", home / "Documents", ROOT):
        if parent.is_dir():
            try:
                for child in parent.iterdir():
                    if child.is_dir():
                        candidates.extend([child / "data", child / "lossy_rag_data"])
            except OSError:
                pass
    seen = set()
    result = []
    for p in candidates:
        key = str(p).casefold()
        if key not in seen:
            seen.add(key)
            result.append(Path(p))
    return result


def prepare_corpus(target):
    # ВАЖНО: запускаем сборщик из репозитория самого исследовательского проекта.
    # Для фиксации версии в научной работе сохраните его SHA256 и коммит GitHub.
    missing = [lib for lib in ("pandas", "pyarrow", "spacy", "huggingface_hub")
               if importlib.util.find_spec(lib) is None]
    if missing:
        raise RuntimeError(
            f"Не установлены библиотеки: {missing}. Выполните в Jupyter: "
            "%pip install pandas pyarrow spacy huggingface_hub; затем Restart Kernel."
        )
    import spacy
    try:
        spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
    except OSError:
        log("Устанавливаю языковую модель spaCy en_core_web_sm...")
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
        spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
    target = Path(target)
    target.mkdir(parents=True, exist_ok=True)
    for filename in ("build_data.py", "questions_paraphrased.csv"):
        out = target / filename
        if not out.exists():
            url = _SOURCE_BASE + filename
            log(f"Получаю {filename} из {url}")
            request = Request(url, headers={"User-Agent": "Lossy-RAG-Jupyter/1.1"})
            try:
                with urlopen(request, timeout=120) as response:
                    contents = response.read(30_000_001)
            except Exception as exc:
                raise RuntimeError(
                    f"Не удалось скачать {filename} из GitHub. "
                    "Можно вручную скачать репозиторий Arnoochka/Lossy-RAG "
                    "и положить data/build_data.py и data/questions_paraphrased.csv "
                    f"в папку {target}."
                ) from exc
            if not contents or len(contents) > 30_000_000:
                raise RuntimeError(f"Неожиданный размер {filename}")
            if filename == "build_data.py":
                try:
                    compile(contents.decode("utf-8-sig"), filename, "exec")
                except (UnicodeDecodeError, SyntaxError) as exc:
                    raise RuntimeError("Из GitHub получен некорректный Python-скрипт") from exc
            if filename == "questions_paraphrased.csv" and b"orig_qid" not in contents[:4096]:
                raise RuntimeError("Неожиданные столбцы файла перефраз")
            temporary = out.with_suffix(out.suffix + ".download")
            temporary.write_bytes(contents)
            temporary.replace(out)
    log(f"Полных таблиц не найдено. Запускаю исходный сборщик SQuAD в {target}")
    log("Сборка данных может потребовать времени; ниже будут сообщения официального скрипта.")
    proc = subprocess.Popen([sys.executable, "-u", "build_data.py"], cwd=str(target),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, errors="replace", bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print("[SQuAD] " + line.rstrip(), flush=True)
    exit_code = proc.wait()
    if exit_code != 0:
        raise RuntimeError(f"Сборщик данных завершился с кодом {exit_code}; см. вывод выше")
    if not prepared_data_exists(target):
        raise RuntimeError(f"Сборка завершилась, но в {target} всё ещё отсутствуют файлы: {_REQUIRED}")
    manifest = {"repo": _SOURCE_BASE,
                "build_script_sha256": file_sha256(target / "build_data.py"),
                "corpus_chunks_sha256": file_sha256(target / "chunks.csv"),
                "data_dir": str(target)}
    json_atomic(target / "auto_build_manifest.json", manifest)
    return target


_folders = [ROOT / "data"]
_found = [folder for folder in _folders if prepared_data_exists(folder)]
if _found:
    # Если встречается тестовый корпус в 350 строк, автоматически не выбираем его.
    _full = [p for p in _found if (p / "chunks.csv").stat().st_size >= 3_000_000]
    if _full:
        DATA_DIR = _full[0]
        log(f"НАЙДЕН готовый корпус: {DATA_DIR}. Повторная загрузка SQuAD не нужна.")
    elif AUTO_PREPARE_DATA:
        log("Найдены только маленькие тестовые корпуса, полный SQuAD будет создан отдельно.")
        DATA_DIR = prepare_corpus(ROOT / "data")
    else:
        raise ValueError("Найден только тестовый корпус, а необходим полный SQuAD")
elif AUTO_PREPARE_DATA:
    DATA_DIR = prepare_corpus(DATA_DIR)
    log(f"Полный корпус подготовлен: {DATA_DIR}")
else:
    raise FileNotFoundError(
        "Файлы chunks.csv/questions.csv/answer_chunks.csv не найдены. "
        "Проверенные папки:\n" + "\n".join(map(str, _folders)) +
        "\nВключите AUTO_PREPARE_DATA=True для автоматической сборки."
    )

# Всё, что ниже, — неизменённая схема, проверка полного корпуса и тех же 500 вопросов.
chunks = pd.read_csv(DATA_DIR / "chunks.csv")
questions = pd.read_csv(DATA_DIR / "questions.csv")
answer_chunks = pd.read_csv(DATA_DIR / "answer_chunks.csv")
for name, df, cols in (
    ("chunks", chunks, ["chunk_id", "text"]),
    ("questions", questions, ["qid", "orig_qid", "role", "question"]),
    ("answer_chunks", answer_chunks, ["orig_qid", "chunk_id", "is_annotated"]),
):
    absent = set(cols) - set(df.columns)
    if absent:
        raise ValueError(f"В {name} нет колонок {sorted(absent)}")
if len(chunks) < MIN_REQUIRED_CHUNKS:
    raise ValueError(f"Сейчас в chunks.csv только {len(chunks):,} фрагментов! Нужен полный корпус SQuAD.")
chunks["chunk_id"] = chunks["chunk_id"].astype("int64")
questions["qid"] = questions["qid"].astype("int64")
questions["orig_qid"] = questions["orig_qid"].astype("int64")
answer_chunks["orig_qid"] = answer_chunks["orig_qid"].astype("int64")
answer_chunks["chunk_id"] = answer_chunks["chunk_id"].astype("int64")
questions["is_paraphrase"] = (
    questions["is_paraphrase"].astype(str).str.lower().isin(["true", "1"])
    if "is_paraphrase" in questions else (questions["qid"] != questions["orig_qid"])
)
answer_chunks["is_annotated"] = answer_chunks["is_annotated"].astype(str).str.lower().isin(["true", "1"])
assert chunks["chunk_id"].is_unique and questions["qid"].is_unique
assert chunks["text"].notna().all() and questions["question"].notna().all()

originals = pd.read_csv(BLOCK1_DIR / 'results/main/selected_train_questions.csv')
paraphrases = pd.read_csv(BLOCK1_DIR / 'results/main/selected_paraphrases.csv') if RUN_PARAPHRASES else questions.iloc[0:0].copy()
for selected in (originals, paraphrases):
    aligned = questions.set_index('qid').reindex(selected.qid)
    if aligned.question.isna().any() or aligned.question.tolist() != selected.question.tolist():
        raise ValueError('Frozen evaluation questions do not match the corpus')

eval_queries = pd.concat([originals, paraphrases], ignore_index=True)
chunk_ids = chunks["chunk_id"].to_numpy(dtype=np.int64)
DOC_HASH = signature(chunk_ids, chunks["text"])
if DOC_HASH != '15704f7b9c5a2a21f5484613e837786e00afc7d596014eb82832dfb0732fb4af':
    raise ValueError('Corpus fingerprint differs from the published experiment; stop and check data')
QUERY_HASH = signature(eval_queries["qid"], eval_queries["question"])
full_question_hash = signature(questions["qid"], questions["question"])

strict_gold = {}
soft_gold = {}
for qid, group in answer_chunks.groupby("orig_qid", sort=False):
    q = int(qid)
    strict_gold[q] = set(group.loc[group.is_annotated, "chunk_id"].astype("int64"))
    soft_gold[q] = set(group["chunk_id"].astype("int64"))
corpus_set = set(chunk_ids.tolist())
no_gold = [int(x) for x in originals.qid if not (strict_gold.get(int(x), set()) & corpus_set)]
if no_gold:
    raise ValueError(f"Нет строгих эталонных фрагментов у {len(no_gold)} вопросов: {no_gold[:5]}")

originals.to_csv(RESULT_DIR / "selected_train_questions.csv", index=False)
if len(paraphrases):
    paraphrases.to_csv(RESULT_DIR / "selected_paraphrases.csv", index=False)
log(f"Полный корпус: {len(chunks):,} документов; исходных вопросов: {len(originals):,}; перефраз: {len(paraphrases):,}")
log("Подготовка данных завершена. До результатов осталось: кэш эмбеддингов → top-k → метрики.")

In [ ]:
# 3. Точный поиск с отображением прогресса и возобновлением после остановки ядра.
def exact_topk_resume(tag, q_unit, doc_unit, doc_ids, query_hash, doc_hash, top_k=None):
    k = int(top_k or max(K_VALUES))
    if len(q_unit) == 0 or len(doc_unit) == 0:
        raise ValueError("Пустая матрица")
    if q_unit.shape[1] != doc_unit.shape[1]:
        raise ValueError("Размерности документов и вопросов отличаются")
    assert k <= len(doc_unit)
    prefix = RESULT_DIR / "rankings" / tag
    path_ids = prefix.with_name(prefix.name + "_topk_ids.npy")
    path_scores = prefix.with_name(prefix.name + "_topk_scores.npy")
    state_path = prefix.with_name(prefix.name + "_search_state.json")
    # Контроль именно вычисленных векторов: прежняя выдача не подменит новые PCA-данные.
    def vectors_hash(x):
        if isinstance(x, np.memmap) and getattr(x, "filename", None):
            return file_sha256(x.filename)
        return hashlib.sha256(np.asarray(x, dtype=np.float32).tobytes()).hexdigest()
    log(f"[{tag}] Проверяю соответствие сохранённых векторов и выдачи ...")
    config = {"tag": tag, "query_hash": query_hash, "doc_hash": doc_hash,
              "representation_docs": vectors_hash(doc_unit),
              "representation_queries": vectors_hash(q_unit),
              "shape_queries": list(q_unit.shape), "shape_docs": list(doc_unit.shape),
              "top_k": k, "version": 4}
    start = 0
    if state_path.exists() and path_ids.exists() and path_scores.exists():
        try:
            st = json.loads(state_path.read_text(encoding="utf-8"))
            i = np.load(path_ids, mmap_mode="r+")
            s = np.load(path_scores, mmap_mode="r+")
            if st["config"] == config and i.shape == (len(q_unit), k) and s.shape == (len(q_unit), k):
                start = int(st["completed_queries"])
                if not 0 <= start <= len(q_unit):
                    raise ValueError('Invalid completed_queries checkpoint')
            else:
                i = s = None
        except (KeyError, ValueError, OSError, json.JSONDecodeError):
            i = s = None
    if start == 0:
        i = np.lib.format.open_memmap(path_ids, mode="w+", dtype=np.int64, shape=(len(q_unit), k))
        s = np.lib.format.open_memmap(path_scores, mode="w+", dtype=np.float32, shape=(len(q_unit), k))
        json_atomic(state_path, {"config": config, "completed_queries": 0})
    log(f"[{tag}] Точный поиск, уже готово: {start}/{len(q_unit)} запросов")
    t0 = time.perf_counter()
    for qb in range(start, len(q_unit), QUERY_BLOCK):
        count = min(QUERY_BLOCK, len(q_unit) - qb)
        qr = np.ascontiguousarray(q_unit[qb:qb+count], dtype=np.float32)
        top_scores = np.full((count, k), -np.inf, dtype=np.float32)
        top_ids = np.full((count, k), -1, dtype=np.int64)
        with threadpool_limits(limits=THREADS):
            for db in range(0, len(doc_unit), DOC_BLOCK):
                doc = np.ascontiguousarray(doc_unit[db:db+DOC_BLOCK], dtype=np.float32)
                sim = qr @ doc.T
                keep = min(k, sim.shape[1])
                inds = stable_topk_positions(sim, doc_ids[db:db+len(doc)], keep)
                ss = np.take_along_axis(sim, inds, axis=1)
                ids = doc_ids[db+inds]
                both_s = np.concatenate([top_scores, ss], axis=1)
                both_ids = np.concatenate([top_ids, ids], axis=1)
                order = np.lexsort((both_ids, -both_s), axis=1)[:, :k]
                top_scores = np.take_along_axis(both_s, order, axis=1)
                top_ids = np.take_along_axis(both_ids, order, axis=1)
        i[qb:qb+count] = top_ids
        s[qb:qb+count] = top_scores
        i.flush(); s.flush()
        done = qb + count
        json_atomic(state_path, {"config": config, "completed_queries": done})
        if qb == start or (qb // QUERY_BLOCK + 1) % PRINT_EVERY_BLOCKS == 0 or done == len(q_unit):
            log(f"[{tag}] готово {done}/{len(q_unit)}; прошло {time.perf_counter()-t0:.1f} c")
    log(f"[{tag}] Поиск завершён; результат на диске.")
    return np.load(path_ids, mmap_mode="r"), np.load(path_scores, mmap_mode="r")


def metrics(tag, table, found, scores, seconds, dimension):
    rows = []
    for idx, row in enumerate(table.itertuples(index=False)):
        gold1 = strict_gold[int(row.orig_qid)]
        gold2 = soft_gold[int(row.orig_qid)]
        for k in K_VALUES:
            seen = set(int(x) for x in found[idx, :k])
            rows.append({"doc_hash": DOC_HASH, "query_hash": QUERY_HASH, "qid": int(row.qid), "orig_qid": int(row.orig_qid),
                         "is_paraphrase": bool(row.is_paraphrase),
                         "k": int(k), "strict_hit": int(bool(seen & gold1)),
                         "soft_hit": int(bool(seen & gold2)), "condition": tag,
                         "dimension": dimension})
    detail = pd.DataFrame(rows)
    if not (detail.strict_hit <= detail.soft_hit).all():
        raise AssertionError("Строгий эталон оказался лучше мягкого: проверьте данные")
    summary = (detail.groupby(["condition", "dimension", "is_paraphrase", "k"], as_index=False)
               .agg(strict_hit=("strict_hit", "mean"),
                    soft_hit=("soft_hit", "mean"), n_questions=("qid", "size")))
    summary["search_seconds_this_run"] = round(float(seconds), 3)
    summary["doc_hash"] = DOC_HASH
    summary["query_hash"] = QUERY_HASH
    detail.to_csv(RESULT_DIR / "tables" / f"{tag}_by_question.csv", index=False)
    summary.to_csv(RESULT_DIR / "tables" / f"{tag}_summary.csv", index=False)
    log(f"[{tag}] Готово! Результаты записаны в tables/{tag}_summary.csv")
    print(summary.loc[~summary.is_paraphrase, ["k", "strict_hit", "soft_hit", "n_questions"]].to_string(index=False), flush=True)
    return detail, summary

# Математическая мини-проверка поиска до запуска моделей
_test_docs = normalize([[1,0,0], [0,1,0], [-1,0,0], [0,0,1]])
_test_queries = normalize([[1,0,0], [0,1,0]])
assert np.allclose(_test_queries @ _test_docs.T, [[1,0,-1,0],[0,1,0,0]])
log("Проверка скалярного произведения и нормировки пройдена")

In [ ]:
# 4. Кэш эмбеддингов с независимым хранением документов и ТОЛЬКО выбранных вопросов.
#    Если есть подтверждённые старые файлы BGE Small, используем их без повторного кодирования.
def safe_cache_paths(slug, kind):
    path = RESULT_DIR / "cache" / f"{slug}_{kind}.npy"
    meta = RESULT_DIR / "cache" / f"{slug}_{kind}.json"
    return path, meta


def load_safe_cache(slug, kind, model_id, fingerprint, nrows):
    path, meta_path = safe_cache_paths(slug, kind)
    if not meta_path.exists():
        return None
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if (meta.get("model_id") == model_id and meta.get("fingerprint") == fingerprint
                and meta.get("kind") == kind):
            source = Path(meta.get("source", str(path)))
            out = checked_vectors(source, nrows, meta.get("dimension"))
            if out is not None and meta.get('array_sha256') == array_hash(out):
                return source
    except (OSError, ValueError, json.JSONDecodeError):
        pass
    return None


def old_bge_small_docs():
    # Portable release vectors: content fingerprint plus exact vector checksum.
    ready = ROOT / 'data/embeddings_bge-small-en-v1.5'
    manifest = ready / 'manifest.json'
    candidate = ready / 'chunk_embeddings.npy'
    if manifest.is_file() and candidate.is_file():
        info = json.loads(manifest.read_text(encoding='utf-8'))
        if (info.get('model_id') == BGE_MODELS['small'] and
            info.get('doc_fingerprint') == DOC_HASH and
            np.array_equal(np.load(ready / 'chunk_ids.npy'), chunk_ids) and
            info['files']['chunk_embeddings.npy']['sha256'] == file_sha256(candidate)):
            if checked_vectors(candidate, len(chunks), 384) is not None:
                return candidate
    # Строгая проверка по SHA256 исходного CSV + порядку chunk_id.
    p = PREVIOUS_RESULTS / "embeddings" / "bge_small_full_docs.npy"
    m = PREVIOUS_RESULTS / "embeddings" / "bge_small_docs_meta.json"
    ids = PREVIOUS_RESULTS / "full_corpus_chunk_ids.csv"
    if not (p.is_file() and m.is_file() and ids.is_file()):
        return None
    try:
        md = json.loads(m.read_text(encoding="utf-8"))
        old_ids = pd.read_csv(ids)["chunk_id"].to_numpy(dtype=np.int64)
        a = checked_vectors(p, len(chunks), 384)
        if (a is not None and md.get("model_id") == BGE_MODELS["small"]
                and md.get("signature") == file_sha256(DATA_DIR / "chunks.csv")
                and np.array_equal(old_ids, chunk_ids)):
            return p
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        pass
    return None


def old_bge_small_queries():
    # Можно использовать прежние 500 запросов, только если совпадают тексты и порядок.
    if len(paraphrases):
        return None
    p = PREVIOUS_RESULTS / "embeddings" / "bge_small_selected_queries.npy"
    m = PREVIOUS_RESULTS / "embeddings" / "bge_small_queries_meta.json"
    old_table = PREVIOUS_RESULTS / "selected_train_questions.csv"
    if not (p.is_file() and m.is_file() and old_table.is_file()):
        return None
    try:
        md = json.loads(m.read_text(encoding="utf-8"))
        old = pd.read_csv(old_table)
        a = checked_vectors(p, len(eval_queries), 384)
        if (a is not None and md.get("model_id") == BGE_MODELS["small"]
                and BGE_QUERY_PREFIX in str(md.get("signature"))
                and list(old["qid"].astype(int)) == list(eval_queries["qid"].astype(int))
                and list(old["question"].astype(str)) == list(eval_queries["question"].astype(str))):
            return p
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        pass
    return None


def old_final_vectors(slug, kind, model_id):
    # Предыдущие финальные кэши другого формата. Пригодны лишь с точной проверкой.
    p = PREVIOUS_FINAL / "embeddings" / f"{slug}_{'documents' if kind == 'docs' else 'questions'}.npy"
    meta = PREVIOUS_FINAL / "embeddings" / f"{slug}_cache.json"
    if not (p.is_file() and meta.is_file()):
        return None
    try:
        md = json.loads(meta.read_text(encoding="utf-8"))
        config = md.get("config", {})
        if config.get("model_id") != model_id:
            return None
        if kind == "docs":
            expected_doc_prefix = NOMIC_DOC_PREFIX if slug == "nomic" else ""
            if (md.get("doc_hash") == DOC_HASH and checked_vectors(p, len(chunks)) is not None
                    and config.get("doc_prefix", expected_doc_prefix) == expected_doc_prefix):
                return p
        else:
            # Старый файл содержит ВСЕ вопросы, не 500. Не путать порядок строк.
            q = checked_vectors(p, len(questions))
            expected_prefix = NOMIC_QUERY_PREFIX if slug == "nomic" else BGE_QUERY_PREFIX
            if (md.get("query_hash") == full_question_hash and q is not None
                    and config.get("query_prefix") == expected_prefix):
                rows = pd.Series(np.arange(len(questions)), index=questions.qid).loc[eval_queries.qid].to_numpy()
                out, _ = safe_cache_paths(slug, kind)
                np.save(out, np.asarray(q[rows], dtype=np.float32))
                return out
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        pass
    return None


def save_cache_manifest(slug, kind, model_id, fingerprint, path, dim):
    _, meta = safe_cache_paths(slug, kind)
    json_atomic(meta, {"kind": kind, "model_id": model_id, "fingerprint": fingerprint,
                       "n_rows": int(len(chunks) if kind == "docs" else len(eval_queries)),
                       "dimension": int(dim), "source": str(path), "created_at": datetime.now().isoformat(),
                       "array_sha256": array_hash(np.load(path, mmap_mode='r'))})


def encode_resume(model, texts, prefix, destination, n_dim, key, nomic=False):
    # Безопасное возобновление по пакетам; временный .partial.npy сохраняется при прерывании.
    part = destination.with_name(destination.stem + ".partial.npy")
    state = destination.with_name(destination.stem + ".partial.json")
    start = 0
    if part.exists() and state.exists():
        try:
            old = json.loads(state.read_text(encoding="utf-8"))
            matrix = np.load(part, mmap_mode="r+")
            if old.get("key") == key and matrix.shape == (len(texts), n_dim):
                start = int(old.get("completed", 0))
                if not 0 <= start <= len(texts):
                    raise ValueError('Invalid encoding checkpoint')
            else:
                del matrix
                start = 0
        except (OSError, ValueError, json.JSONDecodeError):
            start = 0
    if start == 0:
        matrix = np.lib.format.open_memmap(part, mode="w+", dtype=np.float32,
                                            shape=(len(texts), n_dim))
        json_atomic(state, {"key": key, "completed": 0})
    log(f"Векторизация {destination.name}: готово {start}/{len(texts)}")
    pos = start
    batch_size = ENCODE_BATCH_SIZE
    batch_index = 0
    while pos < len(texts):
        batch = [prefix + str(t) for t in texts[pos:pos+batch_size]]
        try:
            raw = model.encode(batch, batch_size=batch_size,
                               convert_to_numpy=True, show_progress_bar=False)
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and batch_size > 1:
                batch_size = max(1, batch_size // 2)
                log(f"Недостаточно памяти GPU при векторизации; размер батча уменьшен до {batch_size}")
                if torch is not None and torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            raise
        v = np.asarray(raw, dtype=np.float32)
        if v.shape != (len(batch), n_dim):
            raise ValueError(f"Ошибка формы эмбеддингов: {v.shape}, ожидалось {(len(batch), n_dim)}")
        v = layer_norm(v) if nomic else normalize(v)
        matrix[pos:pos+len(batch)] = v
        pos += len(batch)
        batch_index += 1
        checkpoint = (batch_index == 1 or batch_index % 25 == 0 or pos == len(texts))
        if checkpoint:
            matrix.flush()
            json_atomic(state, {"key": key, "completed": pos})
        if batch_index == 1 or batch_index % 50 == 0 or pos == len(texts):
            log(f"Векторизация {destination.name}: {pos}/{len(texts)}")
    del matrix
    part.replace(destination)
    state.unlink(missing_ok=True)


def ensure_embeddings(slug, model_id, query_prefix, doc_prefix="", nomic=False):
    docs_path = load_safe_cache(slug, "docs", model_id, DOC_HASH, len(chunks))
    q_path = load_safe_cache(slug, "queries", model_id, QUERY_HASH, len(eval_queries))
    if docs_path is None:
        docs_path = (old_bge_small_docs() if slug == "bge_small" else None) or old_final_vectors(slug, "docs", model_id)
        if docs_path:
            save_cache_manifest(slug, "docs", model_id, DOC_HASH, docs_path, np.load(docs_path, mmap_mode="r").shape[1])
            log(f"[{slug}] Используем готовые документные эмбеддинги: {docs_path}")
    if q_path is None:
        q_path = (old_bge_small_queries() if slug == "bge_small" else None) or old_final_vectors(slug, "queries", model_id)
        if q_path:
            save_cache_manifest(slug, "queries", model_id, QUERY_HASH, q_path, np.load(q_path, mmap_mode="r").shape[1])
            log(f"[{slug}] Используем готовые эмбеддинги 500 вопросов: {q_path}")
    if docs_path is not None and q_path is not None:
        return np.load(docs_path, mmap_mode="r"), np.load(q_path, mmap_mode="r")
    from sentence_transformers import SentenceTransformer
    log(f"[{slug}] Загрузка модели {model_id} (device={DEVICE_USED})")
    # Nomic native implementation requires Transformers >= 5.5 and ST >= 5.3.
    # No deprecated trust_remote_code implementation is used.
    model = SentenceTransformer(model_id, device=DEVICE_USED)
    if nomic:
        print('Nomic preflight: one document and one question...', flush=True)
        test_vectors = model.encode([doc_prefix + 'A short sample document.',
                                     query_prefix + 'What is the sample?'],
                                    convert_to_numpy=True, show_progress_bar=False)
        if np.asarray(test_vectors).shape != (2, 768):
            raise RuntimeError('Unexpected native Nomic embedding shape: '
                               + str(np.asarray(test_vectors).shape))
        log('Nomic preflight PASSED: 768-dimensional vectors; starting full corpus')
    dim = int(model.get_sentence_embedding_dimension())
    if docs_path is None:
        docs_path, _ = safe_cache_paths(slug, "docs")
        encode_resume(model, chunks["text"].tolist(), doc_prefix, docs_path,
                      dim, key=f"{DOC_HASH}|{model_id}|{doc_prefix}|{nomic}|native_hf5", nomic=nomic)
        save_cache_manifest(slug, "docs", model_id, DOC_HASH, docs_path, dim)
    if q_path is None:
        q_path, _ = safe_cache_paths(slug, "queries")
        encode_resume(model, eval_queries["question"].tolist(), query_prefix, q_path,
                      dim, key=f"{QUERY_HASH}|{model_id}|{query_prefix}|{nomic}|native_hf5", nomic=nomic)
        save_cache_manifest(slug, "queries", model_id, QUERY_HASH, q_path, dim)
    del model
    gc.collect()
    if torch is not None and DEVICE_USED == "cuda":
        torch.cuda.empty_cache()
    return np.load(docs_path, mmap_mode="r"), np.load(q_path, mmap_mode="r")

log("Кэш эмбеддингов готов. Проверяются контрольные суммы, а не только названия файлов.")

In [ ]:
# 5. Выполнение исходных моделей. Результат выводится и сохраняется СРАЗУ после каждой.
RUN_MODELS = {
    "small": RUN_BGE_SMALL, "base": RUN_BGE_BASE, "large": RUN_BGE_LARGE
}
all_summaries = []
for size, enabled in RUN_MODELS.items():
    if not enabled:
        log(f"BGE {size} пропущен настройками")
        continue
    tag = f"bge_{size}_full"
    try:
        docs, qs = ensure_embeddings(f"bge_{size}", BGE_MODELS[size], BGE_QUERY_PREFIX)
        if docs.shape[1] != qs.shape[1]:
            raise AssertionError("Размерность кэша вопросов отличается от размерности документов")
        t0 = time.perf_counter()
        ids, sims = exact_topk_resume(tag, normalize(qs), docs, chunk_ids,
                                      QUERY_HASH, DOC_HASH)
        details, summary = metrics(tag, eval_queries, ids, sims,
                                   time.perf_counter()-t0, int(docs.shape[1]))
        all_summaries.append(summary)
        del docs, qs, ids, sims, details
        gc.collect()
    except Exception as e:
        log(f"ОШИБКА на этапе {tag}: {type(e).__name__}: {e}")
        register_error(tag, e)
log("BGE intentionally skipped; existing BGE results remain intact.")

In [ ]:
# 6. PCA — сохраняем матрицу компонент и каждый набор сжатых векторов.
def get_pca_basis(doc_unit, slug, seed, dims):
    maxdim = int(max(dims))
    nfit = min(PCA_FIT_SAMPLE, len(doc_unit))
    if maxdim >= min(nfit, doc_unit.shape[1]):
        raise ValueError("max(PCA_DIMS) должен быть меньше числа обучающих векторов и исходной размерности")
    base = RESULT_DIR / "pca" / f"{slug}_seed{seed}_components{maxdim}"
    comp_path = base.with_name(base.name + ".npz")
    md_path = base.with_name(base.name + ".json")
    cfg = {"slug": slug, "doc_hash": DOC_HASH, "seed": int(seed),
           "fit_n": nfit, "n_components": maxdim, "source_dim": int(doc_unit.shape[1]), "source_vectors_hash": array_hash(doc_unit)}
    if comp_path.exists() and md_path.exists():
        try:
            if json.loads(md_path.read_text(encoding="utf-8")) == cfg:
                z = np.load(comp_path)
                if z["components"].shape == (maxdim, doc_unit.shape[1]):
                    log(f"PCA {slug}, seed={seed}: используем ранее сохранённое преобразование")
                    return np.asarray(z["mean"], dtype=np.float32), np.asarray(z["components"], dtype=np.float32)
        except (OSError, KeyError, ValueError, json.JSONDecodeError):
            pass
    rng = np.random.default_rng(seed)
    selection = np.sort(rng.choice(len(doc_unit), size=nfit, replace=False))
    fit = np.asarray(doc_unit[selection], dtype=np.float32)
    log(f"PCA {slug}, seed={seed}: начало обучения на {nfit} документах, {maxdim} компонентов")
    start = time.perf_counter()
    with threadpool_limits(limits=THREADS):
        p = PCA(n_components=maxdim, svd_solver="randomized", random_state=seed)
        p.fit(fit)
    np.savez(comp_path, components=p.components_.astype(np.float32),
             mean=p.mean_.astype(np.float32), fit_chunk_ids=chunk_ids[selection],
             explained_variance=p.explained_variance_ratio_.astype(np.float32))
    json_atomic(md_path, cfg)
    log(f"PCA {slug}: модель обучена за {time.perf_counter()-start:.1f} с; компоненты сохранены")
    return p.mean_.astype(np.float32), p.components_.astype(np.float32)


def transformed_cache(src, slug, method, dim, kind, seed=42, mean=None, components=None):
    p = RESULT_DIR / "cache" / f"{slug}_{method}_{seed}_{dim}_{kind}.npy"
    metapath = p.with_suffix(".json")
    source_sig = DOC_HASH if kind == "docs" else QUERY_HASH
    basis_sig = (hashlib.sha256(np.asarray(mean).tobytes() + np.asarray(components).tobytes()).hexdigest()
                 if method == "pca" else "matryoshka_no_basis")
    state = {"source_sig": source_sig, "slug": slug, "method": method,
             "dimension": int(dim), "seed": int(seed), "kind": kind,
             "basis_sig": basis_sig, "source_vectors_hash": array_hash(src)}
    a = checked_vectors(p, len(src), dim) if p.exists() else None
    if a is not None and metapath.exists():
        try:
            if json.loads(metapath.read_text(encoding="utf-8")) == state:
                return a
        except (ValueError, OSError, json.JSONDecodeError):
            pass
    temp = p.with_name(p.stem + ".incomplete.npy")
    a = np.lib.format.open_memmap(temp, mode="w+", dtype=np.float32,
                                   shape=(len(src), dim))
    log(f"Проекция: {slug}/{method}/d={dim}/{kind}: 0/{len(src)}")
    for st in range(0, len(src), 4096):
        block = np.asarray(src[st:st+4096], dtype=np.float32)
        if method == "pca":
            if mean is None or components is None:
                raise ValueError("Нет обученной PCA-модели")
            with threadpool_limits(limits=THREADS):
                projected = (block - mean) @ components[:dim].T
        elif method == "matryoshka":
            projected = block[:, :dim]
        else:
            raise ValueError(method)
        a[st:st+len(block)] = normalize(projected)
        if st == 0 or st+len(block) >= len(src) or (st // 4096 + 1) % 5 == 0:
            log(f"Проекция: {slug}/{method}/d={dim}/{kind}: {st+len(block)}/{len(src)}")
    a.flush()
    del a
    temp.replace(p)
    json_atomic(metapath, state)
    return np.load(p, mmap_mode="r")


def run_compression(slug, raw_docs, raw_queries, dims, seeds, with_mrl=False):
    chosen = sorted(set(int(d) for d in dims if 1 <= d < raw_docs.shape[1]))
    if not chosen:
        log(f"Для {slug} нет допустимых размерностей")
        return
    for seed in seeds:
        mean, comps = get_pca_basis(raw_docs, slug, seed, chosen)
        for d in chosen:
            tag = f"{slug}_pca_{d}_seed{seed}"
            try:
                dd = transformed_cache(raw_docs, slug, "pca", d, "docs", seed, mean, comps)
                qq = transformed_cache(raw_queries, slug, "pca", d, "queries", seed, mean, comps)
                t0 = time.perf_counter()
                ids, sims = exact_topk_resume(tag, qq, dd, chunk_ids, QUERY_HASH, DOC_HASH)
                detail, summ = metrics(tag, eval_queries, ids, sims,
                                       time.perf_counter() - t0, d)
                all_summaries.append(summ)
                del dd, qq, ids, sims, detail
                gc.collect()
            except Exception as exc:
                log(f"Ошибка {tag}: {type(exc).__name__}: {exc}")
                log(traceback.format_exc())
                raise
    if with_mrl:
        for d in chosen:
            tag = f"{slug}_matryoshka_{d}"
            dd = transformed_cache(raw_docs, slug, "matryoshka", d, "docs")
            qq = transformed_cache(raw_queries, slug, "matryoshka", d, "queries")
            t0 = time.perf_counter()
            ids, sims = exact_topk_resume(tag, qq, dd, chunk_ids, QUERY_HASH, DOC_HASH)
            detail, summ = metrics(tag, eval_queries, ids, sims,
                                   time.perf_counter() - t0, d)
            all_summaries.append(summ)
            del dd, qq, ids, sims, detail
            gc.collect()

if RUN_PCA_BGE:
    try:
        raw_docs, raw_queries = ensure_embeddings("bge_small", BGE_MODELS["small"], BGE_QUERY_PREFIX)
        run_compression("bge_small", raw_docs, raw_queries, PCA_DIMS, PCA_SEEDS)
        del raw_docs, raw_queries
        gc.collect()
    except Exception as exc:
        register_error("PCA BGE", exc)
log("PCA BGE skipped (Nomic-only run). No BGE files were recalculated.")

In [ ]:
# 7. Отдельно включаемый Nomic: Matryoshka и PCA одной и той же модели.
#    Не включайте этот блок, пока BGE+PCA не получили свои CSV.
def execute_nomic():
    log("Nomic-only experiment: native model, MRL and PCA on the same vectors.")
    docs, queries_nomic = ensure_embeddings("nomic", NOMIC_MODEL,
                                             NOMIC_QUERY_PREFIX, NOMIC_DOC_PREFIX, nomic=True)
    full_doc = transformed_cache(docs, "nomic", "matryoshka",
                                 int(docs.shape[1]), "docs")
    full_q = transformed_cache(queries_nomic, "nomic", "matryoshka",
                               int(queries_nomic.shape[1]), "queries")
    ids, sims = exact_topk_resume("nomic_full", full_q, full_doc, chunk_ids, QUERY_HASH, DOC_HASH)
    det, summ = metrics("nomic_full", eval_queries, ids, sims, 0.0, int(docs.shape[1]))
    all_summaries.append(summ)
    del ids, sims, det, full_doc, full_q
    gc.collect()
    if RUN_PCA_NOMIC:
        run_compression("nomic", docs, queries_nomic, NOMIC_DIMS, PCA_SEEDS, with_mrl=True)
    else:
        for d in [int(x) for x in NOMIC_DIMS if 1 <= int(x) < docs.shape[1]]:
            tag = f"nomic_matryoshka_{d}"
            dd = transformed_cache(docs, "nomic", "matryoshka", d, "docs")
            qq = transformed_cache(queries_nomic, "nomic", "matryoshka", d, "queries")
            ids, sims = exact_topk_resume(tag, qq, dd, chunk_ids, QUERY_HASH, DOC_HASH)
            det, summ = metrics(tag, eval_queries, ids, sims, 0.0, d)
            all_summaries.append(summ)
            del dd, qq, ids, sims, det
    del docs, queries_nomic
    gc.collect()
if RUN_NOMIC:
    try:
        execute_nomic()
    except Exception as exc:
        register_error("Nomic", exc)
        raise  # Do not misleadingly report success on a failed Nomic run.
else:
    log("Nomic пропущен настройками")

In [ ]:
# 8. Сводные результаты, парное сравнение, Bootstrap и тест Мак-Немара
from scipy.stats import binomtest

summary_files = []
for _f in sorted((RESULT_DIR / "tables").glob("*_summary.csv")):
    _df = pd.read_csv(_f)
    if {"doc_hash", "query_hash"}.issubset(_df.columns) and len(_df) and (
       _df["doc_hash"] == DOC_HASH).all() and (_df["query_hash"] == QUERY_HASH).all():
        summary_files.append(_f)
if not summary_files:
    raise RuntimeError("Не создано ни одного файла с метриками: проверьте run_log.txt")
full_summary = pd.concat([pd.read_csv(f) for f in summary_files], ignore_index=True)
full_summary.to_csv(RESULT_DIR / "all_methods_summary.csv", index=False)
show = full_summary.loc[(full_summary.k == 5) & (~full_summary.is_paraphrase.astype(bool))].copy()
print(f"\n=== Сводка по исходным {len(originals)} вопросам (Hit@5) ===")
display(show[["condition", "dimension", "strict_hit", "soft_hit", "n_questions"]]
        .sort_values("strict_hit", ascending=False).reset_index(drop=True))

reference_path = RESULT_DIR / "tables" / "bge_small_full_by_question.csv"
paired = []
if reference_path.exists():
    base = pd.read_csv(reference_path)
    if not (base.doc_hash == DOC_HASH).all() or not (base.query_hash == QUERY_HASH).all():
        raise ValueError("Базовые результаты от другой выборки: перезапустите BGE Small")
    base = base.loc[(base.k == 5) & (~base.is_paraphrase.astype(bool)), ["qid", "strict_hit"]]
    for file in sorted((RESULT_DIR / "tables").glob("*_by_question.csv")):
        if file == reference_path:
            continue
        data = pd.read_csv(file)
        if ("doc_hash" not in data or "query_hash" not in data
                or not (data.doc_hash == DOC_HASH).all()
                or not (data.query_hash == QUERY_HASH).all()):
            continue
        data = data.loc[(data.k == 5) & (~data.is_paraphrase.astype(bool)), ["qid", "strict_hit"]]
        merged = base.merge(data, on="qid", validate="one_to_one", suffixes=("_base", "_alt"))
        if len(merged) != len(base):
            continue
        delta = (merged.strict_hit_alt - merged.strict_hit_base).to_numpy(dtype=np.float64)
        losses = int((delta == -1).sum()); gains = int((delta == 1).sum())
        pval = float(binomtest(min(losses, gains), n=losses+gains, p=0.5).pvalue) if losses+gains else 1.0
        rng = np.random.default_rng(SEED)
        boot = np.empty(2000)
        for j in range(2000):
            boot[j] = delta[rng.integers(0, len(delta), len(delta))].mean()
        ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])
        paired.append({"condition":file.stem.replace("_by_question", ""),
                       "n": len(merged), "delta_hit5":float(delta.mean()),
                       "losses":losses, "gains":gains,
                       "ci95_low":float(ci_lo), "ci95_high":float(ci_hi),
                       "mcnemar_exact_p":pval})
if paired:
    p = pd.DataFrame(paired).sort_values("mcnemar_exact_p").reset_index(drop=True)
    # Holm: скорректированные p с сохранением монотонности
    m = len(p)
    p["holm_p"] = np.maximum.accumulate(np.minimum(1.0, p.mcnemar_exact_p.to_numpy() * (m - np.arange(m))))
    p.to_csv(RESULT_DIR / "paired_vs_bge_small.csv", index=False)
    display(p)

if len(show):
    fig, ax = plt.subplots(figsize=(10, max(4, .43*len(show)+2)))
    sorted_show = show.sort_values("strict_hit")
    ax.barh(sorted_show.condition, sorted_show.strict_hit)
    ax.set_xlabel("Strict Hit@5")
    ax.set_xlim(0, 1)
    ax.grid(axis="x", alpha=.25)
    fig.tight_layout()
    fig.savefig(RESULT_DIR / "figures" / "hit5_by_method.png", dpi=180)
    plt.show()

# Полезно для диагностики без открытия Jupyter:
json_atomic(RESULT_DIR / "validation_report.json", {
    "full_corpus_rows": len(chunks), "original_queries": len(originals),
    "paraphrase_queries": len(paraphrases), "n_conditions": int(show.condition.nunique()),
    "device": DEVICE_USED, "completed_at": datetime.now().isoformat(),
    "notes": "Report reflects completed CSV only; optional runs may be disabled",
})
log("ВСЕ ВКЛЮЧЁННЫЕ ЭТАПЫ ЗАВЕРШЕНЫ. Сводка: all_methods_summary.csv")
print("\nПапка результатов:", RESULT_DIR)
if STAGE_ERRORS:
    log(f"ВНИМАНИЕ: завершено с {len(STAGE_ERRORS)} ошибками отдельных этапов; см. stage_errors.json")
    display(pd.DataFrame(STAGE_ERRORS))
else:
    log("Все включённые модели и эксперименты завершены без ошибок.")

In [ ]:
# 9. Primary research comparison: Matryoshka vs PCA for the same Nomic model.
from scipy.stats import binomtest

comparison_rows = []
for d in sorted(set(NOMIC_DIMS)):
    mrl_file = RESULT_DIR / 'tables' / f'nomic_matryoshka_{d}_by_question.csv'
    pca_file = RESULT_DIR / 'tables' / f'nomic_pca_{d}_seed42_by_question.csv'
    if not mrl_file.is_file() or not pca_file.is_file():
        continue
    mrl = pd.read_csv(mrl_file)
    pca = pd.read_csv(pca_file)
    subset = lambda frame: frame.loc[
        (frame['k'] == 5) & (~frame['is_paraphrase'].astype(bool)),
        ['qid', 'strict_hit', 'doc_hash', 'query_hash']]
    mrl = subset(mrl)
    pca = subset(pca)
    if not (mrl.doc_hash.eq(DOC_HASH).all() and pca.doc_hash.eq(DOC_HASH).all()
            and mrl.query_hash.eq(QUERY_HASH).all() and pca.query_hash.eq(QUERY_HASH).all()):
        raise RuntimeError('Nomic results have different corpus/question hashes')
    paired_df = mrl[['qid', 'strict_hit']].merge(
        pca[['qid', 'strict_hit']], on='qid', validate='one_to_one',
        suffixes=('_matryoshka', '_pca'))
    if len(paired_df) != len(originals):
        raise RuntimeError(f'dimension={d}: expected {len(originals)} original questions, got {len(paired_df)}')
    delta = (paired_df.strict_hit_matryoshka - paired_df.strict_hit_pca).astype(int)
    win = int((delta == 1).sum())
    lose = int((delta == -1).sum())
    p_value = float(binomtest(min(win,lose),n=win+lose,p=0.5).pvalue) if win+lose else 1.0
    comparison_rows.append({
        'dimension': int(d), 'n_questions': len(paired_df),
        'matryoshka_hit5': float(paired_df.strict_hit_matryoshka.mean()),
        'pca_hit5_seed42': float(paired_df.strict_hit_pca.mean()),
        'delta_hit5_mrl_minus_pca': float(delta.mean()),
        'mrl_wins': win, 'pca_wins': lose,
        'mcnemar_exact_p': p_value,
    })
if not comparison_rows:
    raise RuntimeError('No Nomic MRL vs PCA comparisons. Check stage_errors.json and run_log.txt.')
comparison = pd.DataFrame(comparison_rows).sort_values('dimension')
comparison.to_csv(RESULT_DIR / 'nomic_mrl_vs_pca_same_dim.csv', index=False)
print('NOMIC Matryoshka vs PCA, strict Hit@5, same questions:')
display(comparison)
print('SAVED:', RESULT_DIR / 'nomic_mrl_vs_pca_same_dim.csv')


## Finished

The main result is `lossy_rag_final_results_safe/nomic_mrl_vs_pca_same_dim.csv`.
The complete table is `lossy_rag_final_results_safe/all_methods_summary.csv`.
For diagnostics, see `lossy_rag_final_results_safe/run_log.txt`.

**Do not repeat the BGE experiments.** This notebook only runs Nomic + Matryoshka + PCA.
